# AnchorDraw SD1.5 LCM + W03 + L04: Distillation++ lambda sweep

Preliminary paired sweep on the existing smoke8 protocol. The only swept variable is Distillation++ teacher-guidance scale $\lambda$. The implementation follows paper Eq. 9 / Algorithm 1: random re-noising to the next scheduled timestep, teacher CFG 7.5, and one teacher-guided step ($k=1$). `lambda=0` is the unchanged W03 baseline. FID/IS are intentionally excluded for eight samples; this notebook reports CLIP(fg), CLIP(bg), and wall time.

Method audit: the paper states that quantitative results use Eq. 9 clean-estimate interpolation, fully random re-noising, $k=1$, and $\lambda=0.02$ for LCM/LCM-LoRA. The public repository's active `RandomppSolver` branch instead applies the noise-direction guidance term associated with the Eq. 10 approximation while its direct clean interpolation is commented out. This experiment intentionally follows the paper's reported quantitative Algorithm 1, and records that choice in `run_config.json`.

In [ ]:
# 1. Install dependencies without replacing Kaggle's CUDA PyTorch build.
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
packages = ['diffusers>=0.30.0', 'transformers>=4.44.0', 'accelerate', 'peft', 'huggingface_hub', 'safetensors', 'sentencepiece', 'protobuf', 'einops', 'pycocotools', 'open-clip-torch>=2.24.0', 'pandas>=2.0', 'matplotlib']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('[OK] Dependencies installed. If torchao was already imported, restart the session and Run All.')

In [ ]:
# 2. Locate or clone the repository and freeze the paired-sweep protocol.
import os, json, time, gc, types, hashlib, shutil, zipfile, importlib.util
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display, Markdown

REPO_URL = 'https://github.com/GOx9-P/AnchorDraw.git'
WORK_DIR = Path('/kaggle/working')
def is_repo_root(path):
    return (path / 'Ours/src/experiments/semantic_anchor.py').exists() and (path / 'Baseline/semantic-draw-main/src/model/pipeline_semantic_draw.py').exists()
candidates = [Path.cwd(), Path.cwd() / 'AnchorDraw', WORK_DIR / 'AnchorDraw']
REPO_ROOT = next((p.resolve() for p in candidates if is_repo_root(p)), None)
if REPO_ROOT is None:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(WORK_DIR / 'AnchorDraw')], check=True)
    REPO_ROOT = (WORK_DIR / 'AnchorDraw').resolve()
assert is_repo_root(REPO_ROOT), REPO_ROOT

RUN_ID = 'semantic_anchor_sd15_lcm_w03_l04_sigmad4_distillpp_lambda_sweep_smoke8'
RUN_MANIFEST = REPO_ROOT / 'Ours/test_sets/manifests/smoke/coco_val2017_multidiffusion_coco_all_512x512_smoke_bs8.jsonl'
COCO_ROOT = Path(os.environ.get('COCO_ROOT', '/kaggle/working/COCO'))
RUN_ROOT = WORK_DIR / RUN_ID
BY_LAMBDA_DIR = RUN_ROOT / 'generated_images/by_lambda'
BY_SAMPLE_DIR = RUN_ROOT / 'generated_images/by_sample'
GRID_DIR = RUN_ROOT / 'comparison_grids'
METRICS_DIR = RUN_ROOT / 'metrics'
MASK_CACHE_DIR = RUN_ROOT / 'mask_cache'
for p in [BY_LAMBDA_DIR, BY_SAMPLE_DIR, GRID_DIR, METRICS_DIR, MASK_CACHE_DIR]: p.mkdir(parents=True, exist_ok=True)

MODEL_ID = 'runwayml/stable-diffusion-v1-5'
TARGET_SIZE = (512, 512)
RUN_SAMPLES = 8
BATCH_SIZE = 1
BASE_SEED = 2024
BOOTSTRAP_STEPS = 1
MASK_STD, MASK_STRENGTH, PREPROCESS_MASK_COVER_ALPHA = 1.0, 1.0, 0.0
MASK_TYPE, NEGATIVE_PROMPT = 'discrete', ''
ANCHOR_MODE, WEIGHT_POLICY = 'semantic_topk_anchor', 'adaptive_bilateral'
TOPK_ATTENTION_PERCENT, SIGMA_D, SEMANTIC_SIGMA_SCALE = 10.0, 4.0, 1.0
FIXED_LAYER = {'layer_id': 'L04', 'layer_index': 4, 'native_spatial_size': 16, 'layer_name': 'down_blocks.2.attentions.0.transformer_blocks.0.attn2.processor'}
LAMBDA_SWEEP = [0.0, 0.005, 0.01, 0.02, 0.04, 0.08]
TEACHER_CFG_SCALE = 7.5
GUIDE_STEPS = 1
EXPECTED_TIMESTEPS = [999, 919, 759, 499, 259]
for value in LAMBDA_SWEEP: (BY_LAMBDA_DIR / f'lambda_{value:0.3f}').mkdir(parents=True, exist_ok=True)
assert RUN_MANIFEST.exists()
print('[OK] Repo:', REPO_ROOT)
print('[OK] Sweep:', LAMBDA_SWEEP, '| paper LCM/LCM-LoRA value: 0.02')

In [ ]:
# 3. Import project code and download only the COCO files needed by smoke8.
OURS_SRC = REPO_ROOT / 'Ours/src'
BASELINE_SRC = REPO_ROOT / 'Baseline/semantic-draw-main/src'
# Put Ours ahead of Baseline: both expose a top-level module named `data`.
# Purging the cache also makes this cell safe to rerun after a failed import.
for module_name in list(sys.modules):
    if module_name == 'data' or module_name.startswith('data.') or module_name == 'experiments' or module_name.startswith('experiments.'):
        del sys.modules[module_name]
sys.path.insert(0, str(BASELINE_SRC)); sys.path.insert(0, str(OURS_SRC))
from data import COCORegionConfig, build_coco_region_dataloader, batch_item_to_semanticdraw_inputs
from experiments.semantic_anchor import SemanticAnchorCapture, SemanticAnchorRuntime, compute_anchor_measurements, find_target_token_indices
from experiments.distillation_pp import DistillationPPRefiner
from diffusers import UNet2DConditionModel
pipeline_path = BASELINE_SRC / 'model/pipeline_semantic_draw.py'
spec = importlib.util.spec_from_file_location('pipeline_semantic_draw_distillpp', pipeline_path)
pipeline_module = importlib.util.module_from_spec(spec); spec.loader.exec_module(pipeline_module)
SemanticDrawPipeline = pipeline_module.SemanticDrawPipeline

manifest_records = [json.loads(line) for line in RUN_MANIFEST.read_text(encoding='utf-8').splitlines() if line.strip()]
assert len(manifest_records) == RUN_SAMPLES
required_images = [COCO_ROOT / 'val2017' / row['file_name'] for row in manifest_records]
annotations = [COCO_ROOT / 'annotations/instances_val2017.json', COCO_ROOT / 'annotations/captions_val2017.json']
def download(urls, destination):
    if destination.exists() and destination.stat().st_size: return
    destination.parent.mkdir(parents=True, exist_ok=True)
    for url in urls:
        result = subprocess.run(['wget', '-q', '-c', '--no-check-certificate', '-O', str(destination), url])
        if result.returncode == 0 and destination.exists() and destination.stat().st_size: return
    raise RuntimeError(f'Download failed: {destination}')
for image_path in required_images:
    download([f'https://images.cocodataset.org/val2017/{image_path.name}', f'http://images.cocodataset.org/val2017/{image_path.name}'], image_path)
if not all(p.exists() for p in annotations):
    archive = COCO_ROOT / 'annotations_trainval2017.zip'
    download(['https://images.cocodataset.org/annotations/annotations_trainval2017.zip', 'http://images.cocodataset.org/annotations/annotations_trainval2017.zip'], archive)
    with zipfile.ZipFile(archive) as zf: zf.extractall(COCO_ROOT)
assert all(p.exists() for p in [*required_images, *annotations])
print('[OK] Source imports and smoke8 COCO assets ready.')

In [ ]:
# 4. Load the unchanged student and a separate unfused SD1.5 teacher.
def maybe_login_hf():
    token = os.environ.get('HF_TOKEN')
    if token is None:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception: token = None
    if token:
        from huggingface_hub import login
        login(token=token)
def seed_everything(seed):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
def sync_cuda(): torch.cuda.synchronize()
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator.'
maybe_login_hf(); device, dtype = torch.device('cuda:0'), torch.float16
seed_everything(BASE_SEED)
smd = SemanticDrawPipeline(device=device, dtype=dtype, sd_version='1.5', hf_key=MODEL_ID, has_i2t=False, default_mask_std=MASK_STD, default_mask_strength=MASK_STRENGTH, default_preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA, mask_type=MASK_TYPE)
teacher_unet = UNet2DConditionModel.from_pretrained(MODEL_ID, subfolder='unet', torch_dtype=dtype).to(device).eval()
for p in teacher_unet.parameters(): p.requires_grad_(False)
actual_timesteps = [int(t) for t in smd.timesteps.detach().cpu().tolist()]
assert type(smd.scheduler).__name__ == 'LCMScheduler'
assert actual_timesteps == EXPECTED_TIMESTEPS, (actual_timesteps, EXPECTED_TIMESTEPS)
assert FIXED_LAYER['layer_name'] in set(smd.unet.attn_processors)
assert teacher_unet is not smd.unet and teacher_unet.config.in_channels == smd.unet.config.in_channels
print('[OK] Student=SD1.5+LCM-LoRA; teacher=separate base SD1.5; timesteps=', actual_timesteps)

In [ ]:
# 5. Build the identical smoke8 loader and W03/L04 helpers.
config = COCORegionConfig(coco_root=COCO_ROOT, split='val2017', instances_json=annotations[0], captions_json=annotations[1], manifest_path=RUN_MANIFEST, profile='multidiffusion_coco_all', model_family='sd15', target_size=TARGET_SIZE, return_image=True, cache_resized_masks=True, cache_dir=MASK_CACHE_DIR, batch_size=BATCH_SIZE, num_workers=0, pin_memory=False, persistent_workers=False)
loader = build_coco_region_dataloader(config, shuffle=False, drop_last=False)
assert len(loader.dataset) == RUN_SAMPLES
def make_payload(batch, index):
    item = batch_item_to_semanticdraw_inputs(batch, index); meta = item['metadata']
    fg = item['masks'].float().cpu(); bg = (1.0 - fg.sum(dim=0, keepdim=True).clamp(0, 1)).clamp(0, 1)
    prompts = [item['background_prompt'], *item['prompts']]
    return {'sample_id': meta['sample_id'], 'image_id': meta['image_id'], 'prompts': prompts, 'negative_prompts': [NEGATIVE_PROMPT] * len(prompts), 'foreground_prompts': item['prompts'], 'foreground_masks': fg, 'all_masks': torch.cat([bg, fg], dim=0), 'category_names': meta['category_names']}
def select_layer_attention_maps(captured_maps, output_size, layer_name):
    selected = {}
    for key, layer_maps in captured_maps.items():
        matches = [m for m in layer_maps if m.layer_name == layer_name]
        if len(matches) != 1: raise RuntimeError(f'Expected one {layer_name} map at {key}, got {len(matches)}')
        values = F.interpolate(matches[0].values.detach().float().cpu()[None, None], size=output_size, mode='bilinear', align_corners=False)[0, 0]
        selected[key] = (values - values.min()) / (values.max() - values.min()).clamp_min(1e-8)
    return selected
def install_l04(runtime):
    def anchors(self, timestep, foreground_masks, *, strategy, topk_percent, layer_index=None, layer_name=None, **kwargs):
        maps = select_layer_attention_maps(self.attention_capture.maps, self.image_size, FIXED_LAYER['layer_name']); result = []
        for region_index, mask in enumerate(foreground_masks):
            m = compute_anchor_measurements(maps[(int(timestep), region_index)], mask.cpu(), topk_percent=topk_percent)
            prefix = '' if strategy == 'argmax' else 'topk_'
            result.append((float(m[prefix + 'anchor_x']), float(m[prefix + 'anchor_y'])))
        return result
    runtime._anchors_from_current_step = types.MethodType(anchors, runtime)
def run_one(payload, seed, lambda_value, use_hook=True):
    token_indices = [find_target_token_indices(smd.tokenizer, p, c) for p, c in zip(payload['foreground_prompts'], payload['category_names'])]
    refiner = DistillationPPRefiner(teacher_unet=teacher_unet, pipeline=smd, teacher_guidance_scale=lambda_value, guide_steps=GUIDE_STEPS, teacher_cfg_scale=TEACHER_CFG_SCALE) if use_hook else None
    if refiner: refiner.configure_sample(seed)
    seed_everything(seed); sync_cuda(); started = time.perf_counter()
    with SemanticAnchorCapture(smd.unet) as capture:
        capture.configure(token_indices); runtime = SemanticAnchorRuntime(smd, capture, image_size=TARGET_SIZE); install_l04(runtime)
        image, _ = runtime.generate(prompts=payload['prompts'], negative_prompts=payload['negative_prompts'], masks=payload['all_masks'].to(device=device, dtype=torch.float32), foreground_masks=payload['foreground_masks'], mode=ANCHOR_MODE, weight_policy=WEIGHT_POLICY, bootstrap_steps=BOOTSTRAP_STEPS, topk_percent=TOPK_ATTENTION_PERCENT, spatial_sigma_latent=SIGMA_D, semantic_sigma_scale=SEMANTIC_SIGMA_SCALE, mask_stds=MASK_STD, mask_strengths=MASK_STRENGTH, preprocess_mask_cover_alpha=PREPROCESS_MASK_COVER_ALPHA, attention_layer_name=FIXED_LAYER['layer_name'], denoised_refiner=refiner)
    sync_cuda(); elapsed = time.perf_counter() - started
    records = [] if refiner is None else [r.to_dict() for r in refiner.records]
    return image, elapsed, records
print('[OK] Loader and generation helper ready.')

In [ ]:
# 6. Mandatory parity gate: lambda=0 must be pixel-identical to the original W03 path.
first_batch = next(iter(loader)); parity_payload = make_payload(first_batch, 0); parity_seed = BASE_SEED
baseline_image, _, _ = run_one(parity_payload, parity_seed, 0.0, use_hook=False)
zero_image, _, zero_records = run_one(parity_payload, parity_seed, 0.0, use_hook=True)
assert np.array_equal(np.asarray(baseline_image), np.asarray(zero_image)), 'lambda=0 changed W03 pixels'
assert sum(r['applied'] for r in zero_records) == 0
del baseline_image, zero_image, first_batch, parity_payload; gc.collect(); torch.cuda.empty_cache()
print('[OK] lambda=0 is pixel-identical to the unmodified W03/L04 path.')

In [ ]:
# 7. Paired lambda sweep: same sample seed and same private re-noising seed across branches.
generation_rows, distill_rows = [], []
for sample_index, batch in enumerate(loader):
    payload = make_payload(batch, 0); sample_id = payload['sample_id']; seed = BASE_SEED + sample_index
    sample_dir = BY_SAMPLE_DIR / f'{sample_index:04d}_{sample_id}'; sample_dir.mkdir(parents=True, exist_ok=True)
    images = []
    for lambda_value in LAMBDA_SWEEP:
        print(f'[RUN] sample={sample_index + 1}/{RUN_SAMPLES} lambda={lambda_value:.3f}')
        image, elapsed, records = run_one(payload, seed, lambda_value, use_hook=True)
        applied = [r for r in records if r['applied']]
        assert len(applied) == (0 if lambda_value == 0 else GUIDE_STEPS)
        if applied:
            assert applied[0]['step_index'] == 0 and applied[0]['teacher_timestep'] == EXPECTED_TIMESTEPS[1]
            lhs, rhs = applied[0]['refinement_l2'], lambda_value * applied[0]['student_teacher_l2']
            assert abs(lhs - rhs) <= max(2e-4, 0.01 * rhs), (lhs, rhs)
        lambda_dir = BY_LAMBDA_DIR / f'lambda_{lambda_value:0.3f}'
        generated_path = lambda_dir / f'{sample_index:04d}_{sample_id}_generated.png'
        image.save(generated_path); image.save(sample_dir / f'lambda_{lambda_value:0.3f}.png'); images.append((lambda_value, image.copy()))
        generation_rows.append({'index': sample_index, 'sample_id': sample_id, 'image_id': payload['image_id'], 'seed': seed, 'lambda': lambda_value, 'teacher_cfg_scale': TEACHER_CFG_SCALE, 'guide_steps': GUIDE_STEPS, 'method_id': 'WM-03-L04-SIGMAD4-DISTILLPP-EQ9', 'elapsed_sec': elapsed, 'generated_path': str(generated_path)})
        for record in records: distill_rows.append({'sample_index': sample_index, 'sample_id': sample_id, 'lambda': lambda_value, **record})
        del image; gc.collect(); torch.cuda.empty_cache()
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    for ax, (value, image) in zip(axes.flat, images): ax.imshow(image); ax.set_title(f'lambda={value:.3f}'); ax.axis('off')
    fig.suptitle(f'{sample_id} | LCM + W03 + L04 + Distillation++ Eq.9'); fig.tight_layout(); fig.savefig(GRID_DIR / f'{sample_index:04d}_{sample_id}.png', dpi=130); plt.close(fig)
generation_df = pd.DataFrame(generation_rows); distill_df = pd.DataFrame(distill_rows)
assert len(generation_df) == RUN_SAMPLES * len(LAMBDA_SWEEP)
generation_df.to_csv(RUN_ROOT / 'generation_summary.csv', index=False)
(RUN_ROOT / 'generation_summary.json').write_text(json.dumps(generation_rows, ensure_ascii=False, indent=2), encoding='utf-8')
distill_df.to_csv(RUN_ROOT / 'distillationpp_steps.csv', index=False)
with (RUN_ROOT / 'distillationpp_steps.jsonl').open('w', encoding='utf-8') as handle:
    for row in distill_rows: handle.write(json.dumps(row, ensure_ascii=False) + '\n')
print('[OK] Generated', len(generation_df), 'images.')

In [ ]:
# 8. Save method provenance before releasing generation models.
def git_output(*args):
    result = subprocess.run(['git', '-C', str(REPO_ROOT), *args], text=True, capture_output=True)
    return result.stdout.strip() if result.returncode == 0 else None
run_config = {'run_id': RUN_ID, 'purpose': 'preliminary_paired_quality_sweep', 'paper': 'arXiv:2412.08871v1', 'official_code': 'https://github.com/geonyeong-park/inference_distillation', 'official_code_audited_commit': '59dd61e6b2393dd16b8cadadef8897dad22e468a', 'formulation': 'paper Eq.9 clean-estimate interpolation (not the repository noise-direction approximation)', 'student': 'SD1.5 + LCM-LoRA', 'teacher': 'unfused base SD1.5 UNet', 'manifest': str(RUN_MANIFEST), 'samples': RUN_SAMPLES, 'resolution': list(TARGET_SIZE), 'base_seed': BASE_SEED, 'timesteps': actual_timesteps, 'bootstrap_steps': BOOTSTRAP_STEPS, 'anchor_mode': ANCHOR_MODE, 'weight_policy': WEIGHT_POLICY, 'topk_attention_percent': TOPK_ATTENTION_PERCENT, 'fixed_layer': FIXED_LAYER, 'spatial_sigma_latent': SIGMA_D, 'lambda_sweep': LAMBDA_SWEEP, 'paper_lcm_lambda': 0.02, 'teacher_cfg_scale': TEACHER_CFG_SCALE, 'guide_steps': GUIDE_STEPS, 'renoise': 'random at next scheduled timestep', 'teacher_rng': 'private seed = sample seed + 1000003', 'metrics': ['clip_fg', 'clip_bg', 'time'], 'git_commit': git_output('rev-parse', 'HEAD'), 'git_status_short': git_output('status', '--short'), 'semantic_anchor_sha256': hashlib.sha256((OURS_SRC / 'experiments/semantic_anchor.py').read_bytes()).hexdigest(), 'distillation_pp_sha256': hashlib.sha256((OURS_SRC / 'experiments/distillation_pp.py').read_bytes()).hexdigest()}
(RUN_ROOT / 'run_config.json').write_text(json.dumps(run_config, ensure_ascii=False, indent=2), encoding='utf-8')
del teacher_unet, smd, loader; gc.collect(); torch.cuda.empty_cache()
print('[OK] Provenance saved and generation models released.')

In [ ]:
# 9. Preliminary quality metrics. FID/IS require the full evaluation set and are not reported here.
from metrics import MetricEvaluationConfig, run_evaluation, write_metrics_report
metric_rows = []
for lambda_value in LAMBDA_SWEEP:
    branch = generation_df[generation_df['lambda'] == lambda_value].copy()
    branch_summary = BY_LAMBDA_DIR / f'lambda_{lambda_value:0.3f}' / 'generation_summary.json'
    branch_summary.write_text(json.dumps(branch.to_dict('records'), indent=2), encoding='utf-8')
    cfg = MetricEvaluationConfig(manifest_path=RUN_MANIFEST, coco_root=COCO_ROOT, generated_dir=branch_summary.parent, generation_summary=branch_summary, output_dir=METRICS_DIR / f'lambda_{lambda_value:0.3f}', instances_json=annotations[0], captions_json=annotations[1], model_family='sd15', target_size=TARGET_SIZE, metrics=('clip_fg', 'clip_bg', 'time'), batch_size=8, num_workers=0, pin_memory=False, cache_dir=MASK_CACHE_DIR, device='cuda:0', clip_batch_size=16)
    report = run_evaluation(cfg); write_metrics_report(report, cfg.output_dir, prefix=f'lambda_{lambda_value:0.3f}')
    values = report['metrics']; metric_rows.append({'lambda': lambda_value, 'clip_fg_x100': values.get('clip_fg_x100'), 'clip_bg_x100': values.get('clip_bg_x100'), 'time_mean_sec': values.get('time_mean_sec'), 'num_evaluated': report['num_evaluated']})
    del report; gc.collect(); torch.cuda.empty_cache()
metrics_df = pd.DataFrame(metric_rows).sort_values('lambda'); metrics_df.to_csv(RUN_ROOT / 'lambda_metrics_summary.csv', index=False)
display(Markdown('## Preliminary paired sweep results'))
display(metrics_df.style.format({'lambda': '{:.3f}', 'clip_fg_x100': '{:.4f}', 'clip_bg_x100': '{:.4f}', 'time_mean_sec': '{:.3f}'}))

In [ ]:
# 10. Final integrity checks and export.
assert len(list(BY_LAMBDA_DIR.glob('lambda_*/*_generated.png'))) == RUN_SAMPLES * len(LAMBDA_SWEEP)
assert len(list(GRID_DIR.glob('*.png'))) == RUN_SAMPLES
assert set(metrics_df['num_evaluated']) == {RUN_SAMPLES}
assert 0.02 in set(metrics_df['lambda']) and 0.0 in set(metrics_df['lambda'])
tree_lines = [str(path.relative_to(RUN_ROOT)).replace('\\', '/') for path in sorted(RUN_ROOT.rglob('*')) if path.is_file()] + ['artifact_tree.txt']
(RUN_ROOT / 'artifact_tree.txt').write_text('\n'.join(tree_lines) + '\n', encoding='utf-8')
ZIP_PATH = WORK_DIR / f'{RUN_ID}__export.zip'
if ZIP_PATH.exists(): ZIP_PATH.unlink()
shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', root_dir=RUN_ROOT)
print('[OK] Method/code gates passed.')
print('[OK] Export:', ZIP_PATH, '| MB:', round(ZIP_PATH.stat().st_size / 1024**2, 2))
print('[NOTE] Select lambda from CLIP + contact sheets; confirm the selected value later on full1073 with FID/IS/CLIP/Time.')